# Préparation des données FINESS

Ce notebook prépare les données issues du fichier **FINESS Structures – septembre 2026**.

**Objectif :**
- récupérer les informations d’identification des établissements ;
- récupérer leur localisation géographique ;
- préparer une table propre pouvant être reliée à SYGEN.

**Source :** FINESS – extraction mensuelle septembre 2026  
**Clé établissement :** `FI`  
**Clé géographique :** `CODGEO`  
**Sortie :** `finess_structures_clean.csv`

In [5]:
import requests
import ijson
import pandas as pd

In [6]:
path_finess = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/raw/finess-structures-mensuel-202609.json"

output_path = "s3://mangane/Projet_PDS_2026/processed/finess_structures_clean.csv"

In [7]:
def choisir_adresse(adresses):
    if not adresses:
        return {}

    adresses_avec_commune = [
        adresse for adresse in adresses
        if adresse.get("cogCommune") is not None
    ]

    if adresses_avec_commune:
        return adresses_avec_commune[0]

    return adresses[0]

In [8]:
lignes = []

with requests.get(path_finess, stream=True) as r:
    r.raise_for_status()
    r.raw.decode_content = True

    for pmej in ijson.items(r.raw, "pmej.item"):

        infos_pmej = pmej.get("informationsGeneralesPMEJ") or {}
        fi_ej = infos_pmej.get("numFinessPm")

        for ege in pmej.get("ege") or []:

            infos_ege = ege.get("informationsGeneralesEGE") or {}
            adresse = choisir_adresse(ege.get("adresse") or [])
            coordonnees = adresse.get("coordonneesGeographique") or {}

            lignes.append({
                "FI": infos_ege.get("numFinessEge"),
                "FI_EJ": fi_ej,
                "NOM": (
                    infos_ege.get("nomEgeLong")
                    or infos_ege.get("nomEgeCourt")
                ),
                "CATEGORIE": ege.get("categorieentiteGeographiqueExercice"),
                "ETAT": ege.get("etatObjet"),
                "DATE_OUVERTURE": infos_ege.get("dateOuverture"),
                "DATE_FERMETURE": infos_ege.get("dateFermeture"),
                "SIRET": infos_ege.get("siret"),
                "CODGEO": adresse.get("cogCommune"),
                "CODE_POSTAL": adresse.get("codePostal"),
                "ADRESSE": adresse.get("ligneQuatre"),
                "COMMUNE": adresse.get("ligneAcheminement"),
                "LONGITUDE": coordonnees.get("coordonneeX"),
                "LATITUDE": coordonnees.get("coordonneeY"),
                "SCORE_BAN": coordonnees.get("scoreBAN")
            })

df_finess = pd.DataFrame(lignes)

print(df_finess.shape)
df_finess.head()

(174840, 15)


,FI,FI_EJ,NOM,CATEGORIE,ETAT,DATE_OUVERTURE,DATE_FERMETURE,SIRET,CODGEO,CODE_POSTAL,ADRESSE,COMMUNE,LONGITUDE,LATITUDE,SCORE_BAN
0,010780195,010000156,CLINIQUE DOCTEUR CONVERT,365,A,1956-11-16,NaN,77220148900022,01053,01000,62 AVENUE DE JASSERON,BOURG EN BRESSE,5.254203,46.211257,0.9723872727272727
1,010787190,010000164,SMUR POLYCLINIQUE AMBERIEU EN BUGEY ...,321,I,1982-03-04,1993-01-01,NaN,01004,01500,17 RUE AIME VINGTRINIER,AMBERIEU EN BUGEY,NaN,NaN,NaN
2,010780211,010000172,CLINIQUE BEL AIR BOURG EN BRESSE,122,I,1967-07-01,1996-08-29,75920025600015,01053,01000,12 BOULEVARD DE L'HIPPODROME,BOURG EN BRESSE,5.237126,46.191013,0.9684190909090908
3,010002699,010000206,UNITE DIALYSE SITE DU CH DE BELLEY,141,I,2003-05-14,2005-11-17,54692007500043,01034,01300,52 RUE GEORGES GIRERD,BELLEY,5.684769,45.761585,0.9637236363636362
4,010002749,010000206,UNITE AUTODIALYSE SITE CH BELLEY,146,I,2005-05-11,2005-09-07,NaN,01034,01300,54 RUE GEORGES GIRERD,BELLEY,5.684769,45.761585,0.9637236363636362


In [9]:
colonnes_codes = [
    "FI",
    "FI_EJ",
    "CATEGORIE",
    "CODGEO",
    "CODE_POSTAL",
    "SIRET"
]

for col in colonnes_codes:
    df_finess[col] = df_finess[col].astype("string")

for col in ["LONGITUDE", "LATITUDE", "SCORE_BAN"]:
    df_finess[col] = pd.to_numeric(
        df_finess[col],
        errors="coerce"
    )

In [10]:
print("Doublons FI :", df_finess["FI"].duplicated().sum())
print("FI manquants :", df_finess["FI"].isna().sum())

print("\nEtat des structures :")
print(df_finess["ETAT"].value_counts(dropna=False))

print("\nValeurs manquantes :")
print(
    df_finess.isna()
    .sum()
    .sort_values(ascending=False)
)

Doublons FI : 0
FI manquants : 0

Etat des structures :
ETAT
A    104805
I     70035
Name: count, dtype: int64

Valeurs manquantes :
DATE_FERMETURE    104897
SCORE_BAN          55505
LATITUDE           55505
LONGITUDE          55505
SIRET              35408
ADRESSE             9885
COMMUNE             1763
DATE_OUVERTURE        47
CATEGORIE             42
FI_EJ                  0
NOM                    0
ETAT                   0
FI                     0
CODE_POSTAL            0
CODGEO                 0
dtype: int64


In [11]:
df_finess_clean = (
    df_finess
    .sort_values("FI")
    .reset_index(drop=True)
)

print(df_finess_clean.shape)
df_finess_clean.head()

(174840, 15)


,FI,FI_EJ,NOM,CATEGORIE,ETAT,DATE_OUVERTURE,DATE_FERMETURE,SIRET,CODGEO,CODE_POSTAL,ADRESSE,COMMUNE,LONGITUDE,LATITUDE,SCORE_BAN
0,010000024,010780054,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,355,A,1979-02-13,NaN,26010004500012,01451,01440,900 ROUTE DE PARIS,VIRIAT,5.209181,46.222286,0.963911
1,010000032,010780062,CENTRE HOSPITALIER BUGEY SUD,355,A,1901-01-01,NaN,26010003700068,01034,01300,700 AVENUE DE NARVIK,BELLEY,5.679892,45.748331,0.951978
2,010000040,010008407,CENTRE HOSPITALIER NANTUA,355,I,1901-01-01,2007-04-20,26011021800047,01269,01130,50 RUE PAUL PAINLEVE,NANTUA,5.608535,46.148832,0.959462
3,010000057,010008407,CENTRE HOSPITALIER OYONNAX,355,I,1901-01-01,2007-04-20,26011021800013,01283,01100,ROUTE DE VEYZIAT,OYONNAX,5.627348,46.275059,0.954639
4,010000065,010780096,CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER,355,A,1901-01-01,NaN,26010028400017,01427,01606,14 RUE DE L'HOPITAL,TREVOUX CEDEX,4.771957,45.941074,0.956661


In [12]:
df_finess_clean.to_csv(
    output_path,
    index=False
)

print("Fichier sauvegardé :", output_path)

Fichier sauvegardé : s3://mangane/Projet_PDS_2026/processed/finess_structures_clean.csv
